In [2]:
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
)


In [3]:
df = pd.read_csv(r'pipeline_cache\ampds_behavior_context_labeled_features.csv')
df = df.drop(columns=['window_id']) 
print(df.shape)

(5856, 832)


In [4]:
df_reset = df.reset_index(drop=True)

anomaly_df = df_reset[df_reset['is_anomaly'] == 1]
normal_df = df_reset[df_reset['is_anomaly'] == 0]

anom_train, anom_temp = train_test_split(
    anomaly_df, test_size=0.30, stratify=anomaly_df['anomaly_type'], random_state=42
)
anom_val, anom_test = train_test_split(
    anom_temp, test_size=0.50, stratify=anom_temp['anomaly_type'], random_state=42
)

norm_train, norm_temp = train_test_split(normal_df, test_size=0.30, random_state=42)
norm_val, norm_test = train_test_split(norm_temp, test_size=0.50, random_state=42)

train_df = pd.concat([anom_train, norm_train]).sample(frac=1, random_state=42)
val_df   = pd.concat([anom_val, norm_val]).sample(frac=1, random_state=42)
test_df  = pd.concat([anom_test, norm_test]).sample(frac=1, random_state=42)

print(pd.crosstab(
    pd.concat([anom_train, anom_val, anom_test])['anomaly_type'],
    pd.concat([anom_train.assign(split='train'),
               anom_val.assign(split='val'),
               anom_test.assign(split='test')])['split']
))

split                             test  train  val
anomaly_type                                      
appliance_unusual_hours             12     54   12
gradual_drift_decrease               3     16    4
gradual_drift_increase               6     28    6
heating_on_warm_day                  9     44   10
high_usage_low_occupancy            12     54   11
impossible_appliance_combo           5     22    4
multiple_high_power_simultaneous     5     24    5
power_spike                          5     24    5
sensor_glitch                        4     17    3
stuck_appliance_off                 13     61   14
stuck_appliance_on                  14     62   13
sustained_overload                   8     37    8
weekday_pattern_on_weekend           9     43    9
weekend_pattern_on_weekday           9     44   10


C:\Users\revan\AppData\Local\Temp\ipykernel_25732\1503380893.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pd.concat([anom_train.assign(split='train'),
C:\Users\revan\AppData\Local\Temp\ipykernel_25732\1503380893.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  anom_val.assign(split='val'),
C:\Users\revan\AppData\Local\Temp\ipykernel_25732\1503380893.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining

In [5]:
drop_cols = ['is_anomaly', 'anomaly_type']

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['is_anomaly']

X_val = val_df.drop(columns=drop_cols)
y_val = val_df['is_anomaly']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['is_anomaly']

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

print("\nTrain anomaly type counts:\n", train_df['anomaly_type'].value_counts())
print("\nVal anomaly type counts:\n", val_df['anomaly_type'].value_counts())
print("\nTest anomaly type counts:\n", test_df['anomaly_type'].value_counts())


(4098, 830) (4098,)
(879, 830) (879,)
(879, 830) (879,)

Train anomaly type counts:
 anomaly_type
normal                              3568
stuck_appliance_on                    62
stuck_appliance_off                   61
high_usage_low_occupancy              54
appliance_unusual_hours               54
heating_on_warm_day                   44
weekend_pattern_on_weekday            44
weekday_pattern_on_weekend            43
sustained_overload                    37
gradual_drift_increase                28
power_spike                           24
multiple_high_power_simultaneous      24
impossible_appliance_combo            22
sensor_glitch                         17
gradual_drift_decrease                16
Name: count, dtype: int64

Val anomaly type counts:
 anomaly_type
normal                              765
stuck_appliance_off                  14
stuck_appliance_on                   13
appliance_unusual_hours              12
high_usage_low_occupancy             11
heating_on_warm_day  

In [6]:
X_train_normal = X_train[y_train == 0]
print(f"\nNormal training rows: {len(X_train_normal)}")

scaler = MinMaxScaler()
X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_val_scaled = np.clip(scaler.transform(X_val), 0, None)
X_test_scaled = np.clip(scaler.transform(X_test), 0, None)

neg_val = (scaler.transform(X_val) < 0).sum()
neg_test = (scaler.transform(X_test) < 0).sum()
print(f"Clipped negative values — val: {neg_val}, test: {neg_test}")



Normal training rows: 3568
Clipped negative values — val: 205, test: 165


Basic Recon Error

In [13]:
def reconstruction_error_per_sample(X_ori, X_recon):
    return np.linalg.norm(X_ori - X_recon, axis=1)

def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
                       max_iter=1500, tol=1e-3, random_state=42):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_components,
            init="nndsvda",
            solver="cd",
            max_iter=max_iter,
            tol=tol,
            random_state=random_state,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    train_recon = nmf.inverse_transform(nmf.transform(X_train_normal_scaled))
    train_err = reconstruction_error_per_sample(X_train_normal_scaled, train_recon).mean()

    W_val = nmf.transform(X_val_scaled)
    X_val_recon = nmf.inverse_transform(W_val)
    val_scores = reconstruction_error_per_sample(X_val_scaled, X_val_recon)

    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)

    thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))
    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
    for t in thresholds:
        preds = (val_scores >= t).astype(int)
        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

    print(f"K={n_components:>4} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | "
          f"train_err={train_err:.4f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f}")

    return {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "best_threshold": best["threshold"], "best_val_f1": best["f1"],
        "best_val_precision": best["precision"], "best_val_recall": best["recall"],
        "val_scores": val_scores,
    }

In [14]:
n_components_list = [40, 60, 80, 100, 120, 150]

results = []
models = {}
overall_start = time.perf_counter()

for n_comps in n_components_list:
    out = fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components=n_comps)
    results.append({
        "n_components": n_comps, "converged": out["converged"], "n_iter": out["n_iter"],
        "fit_time_sec": round(out["fit_time"], 1), "train_recon_err": out["train_recon_err"],
        "roc_auc": out["roc_auc"], "pr_auc": out["pr_auc"],
        "best_val_f1": out["best_val_f1"], "best_val_precision": out["best_val_precision"],
        "best_val_recall": out["best_val_recall"], "best_threshold": out["best_threshold"],
    })
    models[n_comps] = out

overall_time = time.perf_counter() - overall_start
results_df = pd.DataFrame(results).sort_values(["pr_auc", "roc_auc"], ascending=False)
print("\nValidation results:")
print(results_df.to_string(index=False))
print(f"\nTotal sweep runtime: {overall_time:.1f}s")

# ============================================================
# CELL 8 — pick best model by PR-AUC
# ============================================================
best_k = int(results_df.iloc[0]["n_components"])
best_model = models[best_k]["model"]
best_threshold = models[best_k]["best_threshold"]

print(f"\nBest n_components = {best_k}")
print(f"Best validation threshold = {best_threshold:.6f}")
print(f"Converged: {models[best_k]['converged']} (n_iter={models[best_k]['n_iter']})")

# ============================================================
# CELL 9 — final test evaluation
# ============================================================
W_test = best_model.transform(X_test_scaled)
X_test_recon = best_model.inverse_transform(W_test)
test_scores = reconstruction_error_per_sample(X_test_scaled, X_test_recon)

test_pred = (test_scores >= best_threshold).astype(int)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)

print("\nTest results:")
print(f"ROC AUC:    {test_roc_auc:.4f}")
print(f"PR AUC:     {test_pr_auc:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1:         {test_f1:.4f}")

# ============================================================
# CELL 10 — per-anomaly-type breakdown
# ============================================================
print("\nPer-anomaly-type AUC on test set:")
per_type_results = []
for atype in sorted(test_df['anomaly_type'].unique()):
    if atype == 'normal':
        continue
    mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
    sub = test_df[mask]
    X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
    sub_recon = best_model.inverse_transform(best_model.transform(X_sub_scaled))
    sub_scores = reconstruction_error_per_sample(X_sub_scaled, sub_recon)
    try:
        auc = roc_auc_score(sub['is_anomaly'], sub_scores)
    except ValueError:
        auc = float('nan')
    n_pos = (sub['is_anomaly'] == 1).sum()
    print(f"{atype:35s} AUC={auc:.3f}  n_anomaly={n_pos}")
    per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
print("\n", per_type_df.to_string(index=False))

K=  40 |            OK | n_iter=  185 | fit_time=   2.3s | train_err=1.3829 | roc_auc=0.7902 | pr_auc=0.3969
K=  60 |            OK | n_iter=  844 | fit_time=  41.4s | train_err=0.9775 | roc_auc=0.7932 | pr_auc=0.3682
K=  80 |            OK | n_iter=  336 | fit_time=  27.6s | train_err=0.7583 | roc_auc=0.7972 | pr_auc=0.3868
K= 100 |            OK | n_iter=  301 | fit_time=  36.2s | train_err=0.5453 | roc_auc=0.8022 | pr_auc=0.3950
K= 120 |            OK | n_iter=  653 | fit_time=  57.4s | train_err=0.4249 | roc_auc=0.7877 | pr_auc=0.3898
K= 150 |            OK | n_iter=  510 | fit_time=  93.8s | train_err=0.3333 | roc_auc=0.7881 | pr_auc=0.3636

Validation results:
 n_components  converged  n_iter  fit_time_sec  train_recon_err  roc_auc   pr_auc  best_val_f1  best_val_precision  best_val_recall  best_threshold
           40       True     185           2.3         1.382880 0.790230 0.396907     0.475248            0.545455         0.421053        2.574364
          100       True     

1. Ae-sad

In [17]:
def reconstruction_error_per_sample(X_ori, X_recon):
    return np.sum((X_ori - X_recon) ** 2, axis=1)

def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
                       max_iter=1500, tol=1e-3, random_state=42):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_components,
            init="nndsvda",
            solver="cd",
            max_iter=max_iter,
            tol=tol,
            random_state=random_state,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    train_recon = nmf.inverse_transform(nmf.transform(X_train_normal_scaled))
    train_err = reconstruction_error_per_sample(X_train_normal_scaled, train_recon).mean()

    W_val = nmf.transform(X_val_scaled)
    X_val_recon = nmf.inverse_transform(W_val)
    val_scores = reconstruction_error_per_sample(X_val_scaled, X_val_recon)

    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)

    thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))
    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
    for t in thresholds:
        preds = (val_scores >= t).astype(int)
        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

    print(f"K={n_components:>4} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | "
          f"train_err={train_err:.4f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f}")

    return {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "best_threshold": best["threshold"], "best_val_f1": best["f1"],
        "best_val_precision": best["precision"], "best_val_recall": best["recall"],
        "val_scores": val_scores,
    }

In [18]:
n_components_list = [40, 60, 80, 100, 120, 150]

results = []
models = {}
overall_start = time.perf_counter()

for n_comps in n_components_list:
    out = fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components=n_comps)
    results.append({
        "n_components": n_comps, "converged": out["converged"], "n_iter": out["n_iter"],
        "fit_time_sec": round(out["fit_time"], 1), "train_recon_err": out["train_recon_err"],
        "roc_auc": out["roc_auc"], "pr_auc": out["pr_auc"],
        "best_val_f1": out["best_val_f1"], "best_val_precision": out["best_val_precision"],
        "best_val_recall": out["best_val_recall"], "best_threshold": out["best_threshold"],
    })
    models[n_comps] = out

overall_time = time.perf_counter() - overall_start
results_df = pd.DataFrame(results).sort_values(["pr_auc", "roc_auc"], ascending=False)
print("\nValidation results:")
print(results_df.to_string(index=False))
print(f"\nTotal sweep runtime: {overall_time:.1f}s")

# ============================================================
# CELL 8 — pick best model by PR-AUC
# ============================================================
best_k = int(results_df.iloc[0]["n_components"])
best_model = models[best_k]["model"]
best_threshold = models[best_k]["best_threshold"]

print(f"\nBest n_components = {best_k}")
print(f"Best validation threshold = {best_threshold:.6f}")
print(f"Converged: {models[best_k]['converged']} (n_iter={models[best_k]['n_iter']})")

# ============================================================
# CELL 9 — final test evaluation
# ============================================================
W_test = best_model.transform(X_test_scaled)
X_test_recon = best_model.inverse_transform(W_test)
test_scores = reconstruction_error_per_sample(X_test_scaled, X_test_recon)

test_pred = (test_scores >= best_threshold).astype(int)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)

print("\nTest results:")
print(f"ROC AUC:    {test_roc_auc:.4f}")
print(f"PR AUC:     {test_pr_auc:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1:         {test_f1:.4f}")

# ============================================================
# CELL 10 — per-anomaly-type breakdown
# ============================================================
print("\nPer-anomaly-type AUC on test set:")
per_type_results = []
for atype in sorted(test_df['anomaly_type'].unique()):
    if atype == 'normal':
        continue
    mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
    sub = test_df[mask]
    X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
    sub_recon = best_model.inverse_transform(best_model.transform(X_sub_scaled))
    sub_scores = reconstruction_error_per_sample(X_sub_scaled, sub_recon)
    try:
        auc = roc_auc_score(sub['is_anomaly'], sub_scores)
    except ValueError:
        auc = float('nan')
    n_pos = (sub['is_anomaly'] == 1).sum()
    print(f"{atype:35s} AUC={auc:.3f}  n_anomaly={n_pos}")
    per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
print("\n", per_type_df.to_string(index=False))

K=  40 |            OK | n_iter=  185 | fit_time=   6.9s | train_err=2.1676 | roc_auc=0.7902 | pr_auc=0.3969
K=  60 |            OK | n_iter=  844 | fit_time=  22.1s | train_err=1.1603 | roc_auc=0.7932 | pr_auc=0.3682
K=  80 |            OK | n_iter=  336 | fit_time=  19.2s | train_err=0.7480 | roc_auc=0.7972 | pr_auc=0.3868
K= 100 |            OK | n_iter=  301 | fit_time=  21.8s | train_err=0.4860 | roc_auc=0.8022 | pr_auc=0.3950
K= 120 |            OK | n_iter=  653 | fit_time=  47.0s | train_err=0.3249 | roc_auc=0.7877 | pr_auc=0.3898
K= 150 |            OK | n_iter=  510 | fit_time=  82.4s | train_err=0.2223 | roc_auc=0.7881 | pr_auc=0.3636

Validation results:
 n_components  converged  n_iter  fit_time_sec  train_recon_err  roc_auc   pr_auc  best_val_f1  best_val_precision  best_val_recall  best_threshold
           40       True     185           6.9         2.167588 0.790230 0.396907     0.475248            0.545455         0.421053        6.627596
          100       True     

2. LFR
Investigating recon gap in latent space instead of input space
latent_error = np.linalg.norm(W - W_reconstructed, axis=1)
(not exactly LFR).

In [11]:

def lfr_score(nmf, X):
    W = nmf.transform(X)
    X_recon = nmf.inverse_transform(W)
    W_reconstructed = nmf.transform(X_recon)
    return np.linalg.norm(W - W_reconstructed, axis=1)

def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
                       max_iter=1500, tol=1e-3, random_state=42):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_components,
            init="nndsvda",
            solver="cd",
            max_iter=max_iter,
            tol=tol,
            random_state=random_state,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    train_err = lfr_score(nmf, X_train_normal_scaled).mean()
    val_scores = lfr_score(nmf, X_val_scaled)

    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)


# --------------------------------------------------------

    thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))

    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}

    
    for t in thresholds:
        preds = (val_scores >= t).astype(int)
        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

    print(f"K={n_components:>4} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | "
          f"train_err={train_err:.4f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f}")

    return {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "best_threshold": best["threshold"], "best_val_f1": best["f1"],
        "best_val_precision": best["precision"], "best_val_recall": best["recall"],
        "val_scores": val_scores,
    }

In [12]:

n_components_list = [40, 60, 80, 100, 120, 150]

results = []
models = {}
overall_start = time.perf_counter()

for n_comps in n_components_list:
    out = fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components=n_comps)
    results.append({
        "n_components": n_comps, "converged": out["converged"], "n_iter": out["n_iter"],
        "fit_time_sec": round(out["fit_time"], 1), "train_recon_err": out["train_recon_err"],
        "roc_auc": out["roc_auc"], "pr_auc": out["pr_auc"],
        "best_val_f1": out["best_val_f1"], "best_val_precision": out["best_val_precision"],
        "best_val_recall": out["best_val_recall"], "best_threshold": out["best_threshold"],
    })
    models[n_comps] = out

overall_time = time.perf_counter() - overall_start
results_df = pd.DataFrame(results).sort_values(["pr_auc", "roc_auc"], ascending=False)
print("\nValidation results:")
print(results_df.to_string(index=False))
print(f"\nTotal sweep runtime: {overall_time:.1f}s")

# ============================================================
# pick best model by PR-AUC
# ============================================================
best_k = int(results_df.iloc[0]["n_components"])
best_model = models[best_k]["model"]
best_threshold = models[best_k]["best_threshold"]

print(f"\nBest n_components = {best_k}")
print(f"Best validation threshold = {best_threshold:.6f}")
print(f"Converged: {models[best_k]['converged']} (n_iter={models[best_k]['n_iter']})")

# ============================================================
# final test evaluation — MUST use lfr_score, not raw recon error
# ============================================================
test_scores = lfr_score(best_model, X_test_scaled)
test_pred = (test_scores >= best_threshold).astype(int)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)

print("\nTest results:")
print(f"ROC AUC:    {test_roc_auc:.4f}")
print(f"PR AUC:     {test_pr_auc:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1:         {test_f1:.4f}")

# ============================================================
# per-anomaly-type breakdown — same lfr_score again
# ============================================================
print("\nPer-anomaly-type AUC on test set:")
per_type_results = []
for atype in sorted(test_df['anomaly_type'].unique()):
    if atype == 'normal':
        continue
    mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
    sub = test_df[mask]
    X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
    sub_scores = lfr_score(best_model, X_sub_scaled)
    try:
        auc = roc_auc_score(sub['is_anomaly'], sub_scores)
    except ValueError:
        auc = float('nan')
    n_pos = (sub['is_anomaly'] == 1).sum()
    print(f"{atype:35s} AUC={auc:.3f}  n_anomaly={n_pos}")
    per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
print("\n", per_type_df.to_string(index=False))

K=  40 |            OK | n_iter=  185 | fit_time=   2.4s | train_err=0.0653 | roc_auc=0.5247 | pr_auc=0.1496
K=  60 |            OK | n_iter=  844 | fit_time=  11.7s | train_err=0.0026 | roc_auc=0.4054 | pr_auc=0.1234
K=  80 |            OK | n_iter=  336 | fit_time=   7.9s | train_err=0.0414 | roc_auc=0.3314 | pr_auc=0.1071
K= 100 |            OK | n_iter=  301 | fit_time=  10.8s | train_err=0.0627 | roc_auc=0.4988 | pr_auc=0.1458
K= 120 |            OK | n_iter=  653 | fit_time=  30.4s | train_err=0.0311 | roc_auc=0.4261 | pr_auc=0.1192
K= 150 |            OK | n_iter=  510 | fit_time=  36.5s | train_err=0.0442 | roc_auc=0.5259 | pr_auc=0.1865

Validation results:
 n_components  converged  n_iter  fit_time_sec  train_recon_err  roc_auc   pr_auc  best_val_f1  best_val_precision  best_val_recall  best_threshold
          150       True     510          36.5         0.044243 0.525926 0.186508     0.305882            0.276596         0.342105    6.202987e-02
           40       True     

3. RGAnomaly
Multiple recon signals combined
error = alpha * input_error + (1 - alpha) * latent_error


In [21]:
alpha = 0.5

In [9]:
def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
                       max_iter=1500, tol=1e-3, random_state=42):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_components,
            init="nndsvda",
            solver="cd",
            max_iter=max_iter,
            tol=tol,
            random_state=random_state,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    # train_recon = nmf.inverse_transform(nmf.transform(X_train_normal_scaled))
    # train_err = reconstruction_error_per_sample(X_train_normal_scaled, train_recon).mean()

    # W_val = nmf.transform(X_val_scaled)
    # X_val_recon = nmf.inverse_transform(W_val)
    # val_scores = reconstruction_error_per_sample(X_val_scaled, X_val_recon)

    # LFR: Latent-Space Reconstruction Gap

    W_train = nmf.transform(X_train_normal_scaled)
    X_train_recon = nmf.inverse_transform(W_train)

    # Input-space reconstruction error
    input_error_train = np.linalg.norm(
        X_train_normal_scaled - X_train_recon,
        axis=1
    )



    W_train_reconstructed = nmf.transform(X_train_recon)

    # Latent-space reconstruction error
    latent_error_train = np.linalg.norm(
        W_train - W_train_reconstructed,
        axis=1
    )

    # Combined RGAnomaly error
    train_err = (
        alpha * input_error_train
        + (1 - alpha) * latent_error_train
    ).mean()


    W_val = nmf.transform(X_val_scaled)

    X_val_recon = nmf.inverse_transform(W_val)

    # Input-space reconstruction error
    input_error = np.linalg.norm(
        X_val_scaled - X_val_recon,
        axis=1
    )

    # Reconstruct latent representation
    W_val_reconstructed = nmf.transform(X_val_recon)

    # Latent-space reconstruction error
    latent_error = np.linalg.norm(
        W_val - W_val_reconstructed,
        axis=1
    )

    # RGAnomaly score
    val_scores = (
        alpha * input_error
        + (1 - alpha) * latent_error
    )

# --------------------------------------------------------

    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)

    thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))

    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}

    
    for t in thresholds:
        preds = (val_scores >= t).astype(int)
        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

    print(f"K={n_components:>4} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | "
          f"train_err={train_err:.4f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f}")

    return {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "best_threshold": best["threshold"], "best_val_f1": best["f1"],
        "best_val_precision": best["precision"], "best_val_recall": best["recall"],
        "val_scores": val_scores,
    }

In [10]:
def reconstruction_error_per_sample(X_ori, X_recon):
    return np.sum((X_ori - X_recon) ** 2, axis=1)


def rganomaly_score(nmf, X, alpha):
    """
    alpha * input-space recon error + (1 - alpha) * latent-space recon error.
    """
    W = nmf.transform(X)
    X_recon = nmf.inverse_transform(W)

    # input-space error
    input_error = np.linalg.norm(X - X_recon, axis=1)

    # latent-space error: re-encode the reconstruction, compare to original W
    W_reconstructed = nmf.transform(X_recon)
    latent_error = np.linalg.norm(W - W_reconstructed, axis=1)

    return alpha * input_error + (1 - alpha) * latent_error


def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
                       alpha=0.5, max_iter=1500, tol=1e-3, random_state=42):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_components,
            init="nndsvda",
            solver="cd",
            max_iter=max_iter,
            tol=tol,
            random_state=random_state,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    train_scores = rganomaly_score(nmf, X_train_normal_scaled, alpha)
    train_err = train_scores.mean()

    val_scores = rganomaly_score(nmf, X_val_scaled, alpha)

    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)

    thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))
    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
    for t in thresholds:
        preds = (val_scores >= t).astype(int)
        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

    print(f"K={n_components:>4} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | "
          f"train_err={train_err:.4f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f}")

    return {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "best_threshold": best["threshold"], "best_val_f1": best["f1"],
        "best_val_precision": best["precision"], "best_val_recall": best["recall"],
        "val_scores": val_scores,
        "alpha": alpha,
    }


alphas = [0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
n_components_list = [40, 60, 80, 100, 120, 150]

all_alpha_summaries = []

for alpha in alphas:
    print(f"\n{'='*60}\nAlpha = {alpha}\n{'='*60}")

    results = []
    models = {}
    overall_start = time.perf_counter()

    for n_comps in n_components_list:
        out = fit_and_score_nmf(
            X_train_normal_scaled, X_val_scaled, y_val,
            n_components=n_comps, alpha=alpha
        )
        results.append({
            "n_components": n_comps, "converged": out["converged"], "n_iter": out["n_iter"],
            "fit_time_sec": round(out["fit_time"], 1), "train_recon_err": out["train_recon_err"],
            "roc_auc": out["roc_auc"], "pr_auc": out["pr_auc"],
            "best_val_f1": out["best_val_f1"], "best_val_precision": out["best_val_precision"],
            "best_val_recall": out["best_val_recall"], "best_threshold": out["best_threshold"],
        })
        models[n_comps] = out

    overall_time = time.perf_counter() - overall_start
    results_df = pd.DataFrame(results).sort_values(["pr_auc", "roc_auc"], ascending=False)
    print("\nValidation results:")
    print(results_df.to_string(index=False))
    print(f"\nTotal sweep runtime: {overall_time:.1f}s")

    # ------------------------------------------------------------
    # pick best model by PR-AUC
    # ------------------------------------------------------------
    best_k = int(results_df.iloc[0]["n_components"])
    best_model = models[best_k]["model"]
    best_threshold = models[best_k]["best_threshold"]

    print(f"\nBest n_components = {best_k}")
    print(f"Best validation threshold = {best_threshold:.6f}")
    print(f"Converged: {models[best_k]['converged']} (n_iter={models[best_k]['n_iter']})")

    # ------------------------------------------------------------
    # final test evaluation — MUST use the same scoring fn as training/val
    # ------------------------------------------------------------
    test_scores = rganomaly_score(best_model, X_test_scaled, alpha)
    test_pred = (test_scores >= best_threshold).astype(int)

    test_roc_auc = roc_auc_score(y_test, test_scores)
    test_pr_auc = average_precision_score(y_test, test_scores)
    test_precision = precision_score(y_test, test_pred, zero_division=0)
    test_recall = recall_score(y_test, test_pred, zero_division=0)
    test_f1 = f1_score(y_test, test_pred, zero_division=0)

    print("\nTest results:")
    print(f"ROC AUC:    {test_roc_auc:.4f}")
    print(f"PR AUC:     {test_pr_auc:.4f}")
    print(f"Precision:  {test_precision:.4f}")
    print(f"Recall:     {test_recall:.4f}")
    print(f"F1:         {test_f1:.4f}")

    # ------------------------------------------------------------
    # per-anomaly-type breakdown — same scoring fn again
    # ------------------------------------------------------------
    print("\nPer-anomaly-type AUC on test set:")
    per_type_results = []
    for atype in sorted(test_df['anomaly_type'].unique()):
        if atype == 'normal':
            continue
        mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
        sub = test_df[mask]
        X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
        sub_scores = rganomaly_score(best_model, X_sub_scaled, alpha)
        try:
            auc = roc_auc_score(sub['is_anomaly'], sub_scores)
        except ValueError:
            auc = float('nan')
        n_pos = (sub['is_anomaly'] == 1).sum()
        print(f"{atype:35s} AUC={auc:.3f}  n_anomaly={n_pos}")
        per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

    per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
    print("\n", per_type_df.to_string(index=False))

    all_alpha_summaries.append({
        "alpha": alpha, "best_k": best_k,
        "test_roc_auc": test_roc_auc, "test_pr_auc": test_pr_auc,
        "test_f1": test_f1, "test_precision": test_precision, "test_recall": test_recall,
    })

print(f"\n{'='*60}\nSummary across alphas\n{'='*60}")
summary_df = pd.DataFrame(all_alpha_summaries).sort_values("test_pr_auc", ascending=False)
print(summary_df.to_string(index=False))


Alpha = 0.2
K=  40 |            OK | n_iter=  185 | fit_time=   2.3s | train_err=0.3288 | roc_auc=0.7787 | pr_auc=0.4052
K=  60 |            OK | n_iter=  844 | fit_time=  12.0s | train_err=0.1976 | roc_auc=0.7932 | pr_auc=0.3687
K=  80 |            OK | n_iter=  336 | fit_time=   8.0s | train_err=0.1848 | roc_auc=0.7863 | pr_auc=0.3878
K= 100 |            OK | n_iter=  301 | fit_time=  10.3s | train_err=0.1592 | roc_auc=0.7956 | pr_auc=0.3747
K= 120 |            OK | n_iter=  653 | fit_time=  28.1s | train_err=0.1099 | roc_auc=0.7824 | pr_auc=0.3844
K= 150 |            OK | n_iter=  510 | fit_time=  37.3s | train_err=0.1020 | roc_auc=0.7756 | pr_auc=0.3469

Validation results:
 n_components  converged  n_iter  fit_time_sec  train_recon_err  roc_auc   pr_auc  best_val_f1  best_val_precision  best_val_recall  best_threshold
           40       True     185           2.3         0.328810 0.778730 0.405169     0.484536            0.587500         0.412281        0.572380
           80   